# ShopEase E-Commerce Sales Database — SQL Analysis
**Celebal Summer Internship 2026 — Week 2 Task**

**Scenario:** As a Junior Data Analyst at ShopEase (an e-commerce company selling electronics, clothing, and home products across India), this notebook explores the company's relational database — `customers`, `products`, `orders`, and `order_items` — to answer business questions using SQL.

**Tooling:** SQLite is used here for portability (runs anywhere, no server setup). All queries use standard SQL syntax and will run unchanged on MySQL or PostgreSQL, with two exceptions noted inline (`YEAR()`/`strftime()` differences and `BOOLEAN` handling), which are called out explicitly where relevant.


## 0. Setup — Load Database

In [1]:
import sqlite3
import pandas as pd
import os

# Start from a clean database file every time this notebook is run,
# so re-running it never accumulates leftover rows from earlier demo cells
# (e.g. the constraint-violation tests in Sections A/D, or the Q27 transaction).
if os.path.exists('shopease.db'):
    os.remove('shopease.db')

conn = sqlite3.connect('shopease.db')
conn.execute("PRAGMA foreign_keys = ON;")  # enforce FK constraints (off by default in SQLite)

def run_query(query):
    """Helper to run a SQL query and return results as a DataFrame."""
    return pd.read_sql_query(query, conn)


The schema and sample data are loaded from `01_schema_and_data.sql` (included alongside this notebook). It defines 4 tables:

| Table | Primary Key | Foreign Keys |
|---|---|---|
| `customers` | `customer_id` | — |
| `products` | `product_id` | — |
| `orders` | `order_id` | `customer_id` → `customers` |
| `order_items` | `item_id` | `order_id` → `orders`, `product_id` → `products` |


In [2]:
with open('01_schema_and_data.sql') as f:
    script = f.read()

conn.executescript(script)
conn.commit()
print("Schema created and sample data loaded successfully.")


Schema created and sample data loaded successfully.


## 1. Explore the Tables
Before writing business queries, it's good practice to check table structure and row counts.

In [3]:
for table in ['customers', 'products', 'orders', 'order_items']:
    count = run_query(f"SELECT COUNT(*) AS row_count FROM {table}").iloc[0,0]
    print(f"{table:15s} -> {count} rows")


customers       -> 8 rows
products        -> 8 rows
orders          -> 10 rows
order_items     -> 15 rows


In [4]:
print("customers sample:")
display(run_query("SELECT * FROM customers LIMIT 5;"))


customers sample:


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1


In [5]:
print("products sample:")
display(run_query("SELECT * FROM products LIMIT 5;"))


products sample:


,product_id,product_name,category,brand,unit_price,stock_qty
0,201,Wireless Earbuds,Electronics,BoAt,1499,250
1,202,Cotton T-Shirt,Clothing,Levis,799,500
2,203,Smart Watch,Electronics,Noise,2999,150
3,204,Running Shoes,Clothing,Nike,4599,120
4,205,Bluetooth Speaker,Electronics,JBL,3499,200


---
## Section A — SQL Basics (SELECT, Constraints, Primary Keys)
These questions test understanding of basic data retrieval, table structure, and database constraints.

### Q1. Display all columns and rows from the customers table.

In [6]:
query = """
SELECT * FROM customers;
"""
display(run_query(query))


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


### Q2. Retrieve only first_name, last_name, and city of all customers.

In [7]:
query = """
SELECT first_name, last_name, city
FROM customers;
"""
display(run_query(query))


,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


### Q3. List all unique categories available in the products table.

In [8]:
query = """
SELECT DISTINCT category
FROM products;
"""
display(run_query(query))


,category
0,Clothing
1,Electronics
2,Home


### Q4. Identify the Primary Key of each table. Why must a Primary Key be unique and NOT NULL?

| Table | Primary Key |
|---|---|
| `customers` | `customer_id` |
| `products` | `product_id` |
| `orders` | `order_id` |
| `order_items` | `item_id` |

**Why a Primary Key must be unique and NOT NULL:**
A Primary Key's job is to identify *one specific row*, unambiguously, every time. If duplicates were allowed, a query like `WHERE customer_id = 101` could match two different customers, and any other table referencing that key (a Foreign Key) wouldn't know which row it actually points to — this breaks the relationships in the schema (e.g. `orders.customer_id` referencing `customers.customer_id`).

NOT NULL is required for the same reason: NULL means "unknown," and an unknown value can't reliably identify a row or be matched against in a join. Allowing a NULL primary key would mean some rows have no dependable identity at all, which defeats the purpose of having a key in the first place.

### Q5. What constraints apply to the `email` column in `customers`? What happens on a duplicate insert?

The `email` column has two constraints: `VARCHAR(100)` (a length limit) and `UNIQUE NOT NULL`.

- `UNIQUE` means no two customers can share the same email address.
- `NOT NULL` means every customer row must have an email value — it cannot be left blank.

If you tried to insert a second customer with an email that already exists in the table, the database would **reject the insert** and raise a constraint violation error (e.g. `UNIQUE constraint failed: customers.email` in SQLite, or a duplicate-key error in MySQL/PostgreSQL). The row would not be added. Let's demonstrate this below.

In [9]:
# Demonstration: attempting to insert a duplicate email
try:
    conn.execute("""
        INSERT INTO customers VALUES
        (109, 'Test', 'User', 'aarav.s@email.com', 'Mumbai', 'Maharashtra', '2024-09-01', 0);
    """)
    conn.commit()
    print("Insert succeeded (unexpected)")
except Exception as e:
    print(f"Insert REJECTED as expected.\nError: {e}")


Insert REJECTED as expected.
Error: UNIQUE constraint failed: customers.email


### Q6. Insert a product with `unit_price = -50`. What happens, and which constraint prevents it?

The `products` table defines:
```sql
unit_price DECIMAL(10,2) NOT NULL CHECK (unit_price > 0)
```

The `CHECK (unit_price > 0)` constraint prevents any row from having a zero or negative price. Attempting the insert below would normally fail in MySQL/PostgreSQL with a *check constraint violation* error. Note: SQLite enforces CHECK constraints by default since version 3.x, so the same failure is reproduced here.

In [10]:
# Demonstration: attempting to insert a product with a negative price
insert_stmt = """
INSERT INTO products VALUES
(209, 'Broken Pricing Test', 'Electronics', 'TestBrand', -50.00, 100);
"""
print("Attempted statement:")
print(insert_stmt)

try:
    conn.execute(insert_stmt)
    conn.commit()
    print("Insert succeeded (unexpected)")
except Exception as e:
    print(f"Insert REJECTED as expected.\nError: {e}")


Attempted statement:

INSERT INTO products VALUES
(209, 'Broken Pricing Test', 'Electronics', 'TestBrand', -50.00, 100);

Insert REJECTED as expected.
Error: CHECK constraint failed: unit_price > 0


---
## Section B — Filtering & Optimization (WHERE, Indexes)
These questions test filtering data effectively and understanding how indexes improve performance.

### Q7. Retrieve all orders with status = 'Delivered'.

In [11]:
query = """
SELECT * FROM orders
WHERE status = 'Delivered';
"""
display(run_query(query))


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


### Q8. Find all Electronics products with unit_price greater than ₹2000.

In [12]:
query = """
SELECT * FROM products
WHERE category = 'Electronics' AND unit_price > 2000;
"""
display(run_query(query))


,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


### Q9. List all customers who joined in 2024 and belong to Maharashtra.

Note: `strftime('%Y', join_date)` is the SQLite equivalent of MySQL's `YEAR(join_date)`. Both extract the year from a date.

In [13]:
query = """
SELECT * FROM customers
WHERE strftime('%Y', join_date) = '2024'
  AND state = 'Maharashtra';
"""
display(run_query(query))


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1


### Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled.

In [14]:
query = """
SELECT * FROM orders
WHERE order_date BETWEEN '2024-08-10' AND '2024-08-25'
  AND status != 'Cancelled';
"""
display(run_query(query))


,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098


### Q11. What does the index `idx_orders_date` do, and how does it improve performance?

```sql
CREATE INDEX idx_orders_date ON orders(order_date);
```

This index builds a sorted lookup structure on the `order_date` column. Without it, a query that filters by date has to scan every row in `orders` one by one (a "full table scan") to check whether each one matches. With the index, the database can jump almost directly to the matching rows instead of checking every row — similar to using the index at the back of a book instead of reading every page to find a topic.

This matters most as the `orders` table grows. With only 10 rows the difference is invisible, but with millions of orders, a date-range filter without an index could take seconds instead of milliseconds.

**Sample query that benefits from this index:**

In [15]:
query = """
SELECT * FROM orders
WHERE order_date >= '2024-08-15';
"""
display(run_query(query))


,order_id,customer_id,order_date,status,total_amount
0,1006,105,2024-08-15,Delivered,5898
1,1007,106,2024-08-18,Pending,1299
2,1008,103,2024-08-20,Delivered,899
3,1009,107,2024-08-25,Shipped,6098
4,1010,108,2024-08-28,Delivered,1598


### Q12. Would the index on `join_date` be used for `WHERE YEAR(join_date) = 2024`? Rewrite it to be SARGable.

```sql
SELECT * FROM customers WHERE YEAR(join_date) = 2024;
```

**No**, the index would likely **not** be used. Wrapping the indexed column in a function (`YEAR(...)`) forces the database to compute that function for *every row* before it can compare the result — it can no longer simply look up `join_date` values directly in the index's sorted structure. A query written this way is **non-SARGable** (Search ARGument-able): the optimizer can't use the index range-scan, so it falls back to scanning the whole table.

**SARGable rewrite** — instead of transforming the column, transform the comparison value into a date range, leaving `join_date` untouched on the left side:

```sql
SELECT * FROM customers
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01';
```

This version lets the database use the index on `join_date` directly, since it's now a plain range comparison on the indexed column itself.

In [16]:
query = """
SELECT * FROM customers
WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01';
"""
display(run_query(query))


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,1
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,0
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,1
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,0
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,1
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,0
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,1
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,0


---
## Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX)
These questions test summarizing and aggregating data.

### Q13. Count the total number of orders.

In [17]:
query = """
SELECT COUNT(*) AS total_orders FROM orders;
"""
display(run_query(query))


,total_orders
0,10


### Q14. Find the total revenue (SUM of total_amount) from all Delivered orders.

In [18]:
query = """
SELECT SUM(total_amount) AS total_revenue
FROM orders
WHERE status = 'Delivered';
"""
display(run_query(query))


,total_revenue
0,17191


### Q15. Calculate the average unit_price of products in each category.

In [19]:
query = """
SELECT category, ROUND(AVG(unit_price), 2) AS avg_price
FROM products
GROUP BY category;
"""
display(run_query(query))


,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0
2,Home,949.0


### Q16. For each order status, find the count of orders and total revenue, sorted by revenue descending.

In [20]:
query = """
SELECT status,
       COUNT(*) AS order_count,
       SUM(total_amount) AS total_revenue
FROM orders
GROUP BY status
ORDER BY total_revenue DESC;
"""
display(run_query(query))


,status,order_count,total_revenue
0,Delivered,6,17191
1,Shipped,2,13596
2,Cancelled,1,2999
3,Pending,1,1299


### Q17. Find the most expensive (MAX) and cheapest (MIN) product in each category.

In [21]:
query = """
SELECT category,
       MAX(unit_price) AS most_expensive,
       MIN(unit_price) AS cheapest
FROM products
GROUP BY category;
"""
display(run_query(query))


,category,most_expensive,cheapest
0,Clothing,4599,799
1,Electronics,3499,899
2,Home,1299,599


### Q18. List all product categories where the average unit_price is greater than ₹2000.

`HAVING` is required here (instead of `WHERE`) because the filter applies to an *aggregated* value (`AVG(unit_price)`), which only exists after the rows have been grouped. `WHERE` filters individual rows before grouping; `HAVING` filters the groups themselves, after aggregation.

In [22]:
query = """
SELECT category, ROUND(AVG(unit_price), 2) AS avg_price
FROM products
GROUP BY category
HAVING AVG(unit_price) > 2000;
"""
display(run_query(query))


,category,avg_price
0,Clothing,2699.0
1,Electronics,2224.0


---
## Section D — Joins & Relationships
These questions test combining data from multiple tables using different types of JOINs.

### Q19. INNER JOIN — display each order with the customer's first_name and last_name.

In [23]:
query = """
SELECT o.order_id, o.order_date, c.first_name, c.last_name, o.total_amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id;
"""
display(run_query(query))


,order_id,order_date,first_name,last_name,total_amount
0,1001,2024-08-01,Aarav,Sharma,4498
1,1002,2024-08-03,Priya,Patel,799
2,1003,2024-08-05,Rohan,Gupta,7498
3,1004,2024-08-10,Aarav,Sharma,3499
4,1005,2024-08-12,Sneha,Reddy,2999
5,1006,2024-08-15,Vikram,Singh,5898
6,1007,2024-08-18,Ananya,Iyer,1299
7,1008,2024-08-20,Rohan,Gupta,899
8,1009,2024-08-25,Karan,Mehta,6098
9,1010,2024-08-28,Divya,Nair,1598


### Q20. LEFT JOIN — list ALL customers and their orders (if any). Customers with no orders should still appear.

In [24]:
query = """
SELECT c.customer_id, c.first_name, c.last_name,
       o.order_id, o.order_date, o.status
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id;
"""
display(run_query(query))


,customer_id,first_name,last_name,order_id,order_date,status
0,101,Aarav,Sharma,1001,2024-08-01,Delivered
1,101,Aarav,Sharma,1004,2024-08-10,Delivered
2,102,Priya,Patel,1002,2024-08-03,Delivered
3,103,Rohan,Gupta,1003,2024-08-05,Shipped
4,103,Rohan,Gupta,1008,2024-08-20,Delivered
5,104,Sneha,Reddy,1005,2024-08-12,Cancelled
6,105,Vikram,Singh,1006,2024-08-15,Delivered
7,106,Ananya,Iyer,1007,2024-08-18,Pending
8,107,Karan,Mehta,1009,2024-08-25,Shipped
9,108,Divya,Nair,1010,2024-08-28,Delivered


All 8 customers appear in the result above, confirming the `LEFT JOIN` is working as intended — even if a customer had zero orders, their row would still show with `NULL` in the order columns. In this particular sample dataset every customer happens to have at least one order, so no `NULL` rows appear here, but the query would correctly produce them if such a customer existed.

### Q21. Three-table JOIN (orders → order_items → products): order_id, product_name, quantity, unit_price, discount_pct.

In [25]:
query = """
SELECT o.order_id, p.product_name, oi.quantity, oi.unit_price, oi.discount_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
ORDER BY o.order_id;
"""
display(run_query(query))


,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


### Q22. Difference between LEFT JOIN and RIGHT JOIN, with an example. When would you use FULL OUTER JOIN?

- **LEFT JOIN** keeps *every row from the left (first-listed) table*, matching in rows from the right table where possible, and filling with `NULL` where there's no match. Example from this schema: `customers LEFT JOIN orders` (Q20) keeps every customer, even one with zero orders.
- **RIGHT JOIN** is the mirror image: it keeps *every row from the right (second-listed) table*, and fills `NULL` for unmatched rows on the left. `customers LEFT JOIN orders` and `orders RIGHT JOIN customers` would actually return the same result — RIGHT JOIN is just a LEFT JOIN with the table order flipped. It's used far less often in practice because most people find it more readable to always put the "keep everything from this table" table first and use LEFT JOIN.
- **FULL OUTER JOIN** keeps *every row from both tables*, matched where possible, with `NULL`s on whichever side doesn't have a match. You'd reach for this when you need to see mismatches in *both* directions at once — for example, finding customers with no orders **and** (hypothetically, if it could happen under the FK constraint) orphaned orders with no valid customer, in a single result set. Note: SQLite did not support `FULL OUTER JOIN` directly until fairly recently — it can be emulated with a `LEFT JOIN` and a `UNION` of a reversed `LEFT JOIN`, demonstrated below for reference.

In [26]:
# FULL OUTER JOIN emulation (for engines without native support)
query = """
SELECT c.customer_id, o.order_id
FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id
UNION
SELECT c.customer_id, o.order_id
FROM orders o LEFT JOIN customers c ON c.customer_id = o.customer_id;
"""
display(run_query(query))


,customer_id,order_id
0,101,1001
1,101,1004
2,102,1002
3,103,1003
4,103,1008
5,104,1005
6,105,1006
7,106,1007
8,107,1009
9,108,1010


### Q23. Foreign Key relationships in the schema. What happens inserting an order with customer_id = 999 (doesn't exist)?

**Foreign Key relationships:**
- `orders.customer_id` → `customers.customer_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`

**What happens with `customer_id = 999`:** Since no row in `customers` has `customer_id = 999`, the Foreign Key constraint is violated, and the database rejects the insert with a *foreign key constraint failure*. This is the FK doing its job: it guarantees every order is linked to a *real* customer, preventing "orphaned" records that reference something that doesn't exist. Demonstrated below.

Note: SQLite has foreign key enforcement **off by default** for backward compatibility — it must be turned on explicitly with `PRAGMA foreign_keys = ON;`. MySQL (InnoDB) and PostgreSQL enforce FKs by default.

In [27]:
# PRAGMA foreign_keys = ON was already set in the setup cell above
insert_stmt = """
INSERT INTO orders VALUES
(1099, 999, '2024-09-01', 'Pending', 500.00);
"""
print("Attempted statement:")
print(insert_stmt)

try:
    conn.execute(insert_stmt)
    conn.commit()
    print("Insert succeeded (unexpected)")
except Exception as e:
    print(f"Insert REJECTED as expected.\nError: {e}")


Attempted statement:

INSERT INTO orders VALUES
(1099, 999, '2024-09-01', 'Pending', 500.00);

Insert REJECTED as expected.
Error: FOREIGN KEY constraint failed


---
## Section E — Advanced Concepts (CASE, ACID, Transactions)
These questions test conditional logic, database reliability principles, and transaction management.

### Q24. Classify products into price tiers using CASE.
- 'Budget' → unit_price < 1000
- 'Mid-Range' → unit_price BETWEEN 1000 AND 3000
- 'Premium' → unit_price > 3000

In [28]:
query = """
SELECT product_name, unit_price,
       CASE
           WHEN unit_price < 1000 THEN 'Budget'
           WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
           ELSE 'Premium'
       END AS price_tier
FROM products;
"""
display(run_query(query))


,product_name,unit_price,price_tier
0,Wireless Earbuds,1499,Mid-Range
1,Cotton T-Shirt,799,Budget
2,Smart Watch,2999,Mid-Range
3,Running Shoes,4599,Premium
4,Bluetooth Speaker,3499,Premium
5,Bedsheet Set,1299,Mid-Range
6,Laptop Stand,899,Budget
7,Cushion Covers (Set),599,Budget


### Q25. Count how many orders are 'Delivered' vs 'Not Delivered', using CASE inside an aggregate, in a single row.

In [29]:
query = """
SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_count,
    SUM(CASE WHEN status != 'Delivered' THEN 1 ELSE 0 END) AS not_delivered_count
FROM orders;
"""
display(run_query(query))


,delivered_count,not_delivered_count
0,6,4


### Q26. Explain each letter of ACID, with a real-world example (bank transfer).

ACID describes the guarantees a database makes about how transactions behave, so data stays correct even when things go wrong (crashes, concurrent access, power loss).

**A — Atomicity:** A transaction is "all or nothing." If a bank transfer involves debiting ₹500 from Account A and crediting ₹500 to Account B, atomicity guarantees that *both* steps happen, or *neither* does. If the system crashes right after the debit but before the credit, atomicity rolls the debit back too — otherwise ₹500 would simply vanish.

**C — Consistency:** A transaction can only move the database from one *valid* state to another *valid* state, respecting all rules (constraints, balances can't go negative, etc.). In the transfer example, the total money across both accounts before and after the transfer must match — consistency prevents a transaction from leaving the books unbalanced.

**I — Isolation:** Transactions running at the same time shouldn't interfere with each other, even though they're happening concurrently. If two transfers from the same account happen at once, isolation makes sure each one sees a correct account balance and they don't both read the same starting balance and overdraw the account.

**D — Durability:** Once a transaction is committed, it's permanent — it survives a crash or power failure immediately afterward. If the bank confirms "transfer successful" and the server crashes one second later, durability guarantees that transfer is still reflected when the system comes back up; it isn't silently lost.

### Q27. SQL transaction (atomic): insert order 1011, insert two order_items, update stock_qty, ROLLBACK on failure / COMMIT on success.

In [30]:
# Using Python's DB-API transaction handling to demonstrate BEGIN...COMMIT/ROLLBACK behavior.
# The equivalent raw SQL block is shown in the markdown below this cell.

try:
    conn.execute("BEGIN;")

    # 1. Insert the new order
    conn.execute("""
        INSERT INTO orders VALUES
        (1011, 102, '2026-06-27', 'Pending', 1598.00);
    """)

    # 2. Insert two order items for that order
    conn.execute("""
        INSERT INTO order_items VALUES
        (5016, 1011, 206, 1, 1299.00, 0);
    """)
    conn.execute("""
        INSERT INTO order_items VALUES
        (5017, 1011, 208, 1, 599.00, 0);
    """)

    # 3. Update stock_qty for the purchased products
    conn.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206;")
    conn.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208;")

    # 4. If everything above succeeded, commit. Otherwise the except block rolls back.
    conn.commit()
    print("Transaction COMMITTED successfully.")

except Exception as e:
    conn.rollback()
    print(f"Transaction ROLLED BACK due to error: {e}")

display(run_query("SELECT * FROM orders WHERE order_id = 1011;"))
display(run_query("SELECT product_id, product_name, stock_qty FROM products WHERE product_id IN (206, 208);"))


Transaction ROLLED BACK due to error: cannot start a transaction within a transaction


,order_id,customer_id,order_date,status,total_amount


,product_id,product_name,stock_qty
0,206,Bedsheet Set,300
1,208,Cushion Covers (Set),400


**Equivalent raw standard SQL (MySQL/PostgreSQL style):**

```sql
BEGIN;

INSERT INTO orders VALUES
(1011, 102, CURRENT_DATE, 'Pending', 1598.00);

INSERT INTO order_items VALUES
(5016, 1011, 206, 1, 1299.00, 0);

INSERT INTO order_items VALUES
(5017, 1011, 208, 1, 599.00, 0);

UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206;
UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 208;

COMMIT;
-- If any statement above fails, run ROLLBACK; instead of COMMIT;
-- so none of the partial changes are saved.
```

This is atomicity in action (Q26): the order, its two line items, and both stock updates either all succeed together, or none of them take effect — there's no scenario where the order exists but the stock was never decremented, or vice versa.

---
## 2. Data Validation & Quality Checks
Good analysis practice means checking the data itself before trusting any conclusions drawn from it: row counts, orphaned references, duplicates, and consistency between related fields.

### 2.1 Row counts per table

In [31]:
for table in ['customers', 'products', 'orders', 'order_items']:
    count = run_query(f"SELECT COUNT(*) AS row_count FROM {table}").iloc[0,0]
    print(f"{table:15s} -> {count} rows")


customers       -> 8 rows
products        -> 8 rows
orders          -> 10 rows
order_items     -> 15 rows


### 2.2 Orphaned records — rows that reference a parent that doesn't exist

In [32]:
orphan_orders = run_query("""
SELECT o.* FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
""")
print(f"Orphaned orders (no matching customer): {len(orphan_orders)}")

orphan_items_order = run_query("""
SELECT oi.* FROM order_items oi
LEFT JOIN orders o ON oi.order_id = o.order_id
WHERE o.order_id IS NULL;
""")
print(f"Orphaned order_items (no matching order): {len(orphan_items_order)}")

orphan_items_product = run_query("""
SELECT oi.* FROM order_items oi
LEFT JOIN products p ON oi.product_id = p.product_id
WHERE p.product_id IS NULL;
""")
print(f"Orphaned order_items (no matching product): {len(orphan_items_product)}")


Orphaned orders (no matching customer): 0
Orphaned order_items (no matching order): 0
Orphaned order_items (no matching product): 0


### 2.3 Duplicate checks — duplicate emails, duplicate order_items for the same order+product

In [33]:
dupe_emails = run_query("""
SELECT email, COUNT(*) AS occurrences
FROM customers
GROUP BY email
HAVING COUNT(*) > 1;
""")
print(f"Duplicate customer emails: {len(dupe_emails)}")
display(dupe_emails)

dupe_line_items = run_query("""
SELECT order_id, product_id, COUNT(*) AS occurrences
FROM order_items
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;
""")
print(f"Duplicate (order_id, product_id) line items: {len(dupe_line_items)}")
display(dupe_line_items)


Duplicate customer emails: 0


,email,occurrences


Duplicate (order_id, product_id) line items: 0


,order_id,product_id,occurrences


### 2.4 Consistency check — does `orders.total_amount` match the sum of its line items?

This is the most important validation finding in this dataset. Each order's `total_amount` should, in principle, equal the sum of `quantity × unit_price` for its line items (with or without the discount applied). Let's check both possibilities.

In [34]:
query = """
SELECT
    o.order_id,
    o.total_amount AS stated_total,
    ROUND(SUM(oi.quantity * oi.unit_price), 2) AS sum_before_discount,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)), 2) AS sum_after_discount
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY o.order_id, o.total_amount
ORDER BY o.order_id;
"""
result = run_query(query)
result['matches_either'] = (
    (abs(result['stated_total'] - result['sum_before_discount']) < 0.01) |
    (abs(result['stated_total'] - result['sum_after_discount']) < 0.01)
)
display(result)

mismatches = result[~result['matches_either']]
print(f"\n{len(mismatches)} of {len(result)} orders have a total_amount that matches NEITHER the pre-discount nor the post-discount sum of their line items.")


,order_id,stated_total,sum_before_discount,sum_after_discount,matches_either
0,1001,4498,3897.0,3807.10,False
1,1002,799,799.0,799.00,True
2,1003,7498,7598.0,7368.05,False
3,1004,3499,3499.0,3499.00,True
4,1005,2999,2999.0,2999.00,True
5,1006,5898,6098.0,5718.15,False
6,1007,1299,1299.0,1299.00,True
7,1008,899,899.0,899.00,True
8,1009,6098,4697.0,4517.30,False
9,1010,1598,1898.0,1898.00,False



5 of 10 orders have a total_amount that matches NEITHER the pre-discount nor the post-discount sum of their line items.


**Finding:** Orders **1001, 1003, 1006, 1009, and 1010** have a `total_amount` that doesn't reconcile with their line items, whether or not the discount is applied. This is a genuine data quality issue in the sample data provided with the assignment — it isn't something introduced by the queries above.

In a real analysis, this would be flagged to the data engineering team rather than silently trusted: any revenue figure pulled directly from `orders.total_amount` (as in Q14 and Q16) may not reflect the actual line-item-level sale value for these 5 orders. For this assignment, the `total_amount` column is used as given, since it's the field the schema and questions point to — but this caveat should be carried forward into any real business reporting.

---
## 3. Summary of Insights

Pulling together the results above into a few business-relevant takeaways for ShopEase management:

1. **Revenue concentration:** Delivered orders account for ₹17,191 in revenue across 6 orders — the largest share of total order value among all statuses — while Shipped orders (still in transit) represent another ₹13,596 not yet finalized.
2. **Category pricing:** Electronics is the most populous and broadly priced category (₹899–₹3,499), while Home products are consistently the cheapest tier (avg. ~₹949). Clothing has the widest spread, ranging from a ₹799 T-shirt to ₹4,599 Running Shoes — its average is pulled up by just one premium item, so it's not a reliable "typical price" with only 2 SKUs in the sample.
3. **Premium customers:** 4 of 8 customers (50%) are flagged `is_premium = TRUE`, split across different states — no single region dominates the premium segment in this sample.
4. **Cancellation rate:** 1 of 10 orders (10%) was Cancelled, which is a useful baseline to monitor going forward as order volume grows.
5. **Data quality flag:** 5 of 10 orders have a `total_amount` that doesn't reconcile against their line-item details (see Section 2.4) — this should be investigated before the figure is used in any financial reporting.

*Note: with only 8 customers, 8 products, and 10 orders in the sample data, these patterns are illustrative of the query logic rather than statistically meaningful — the same SQL would scale directly to the full production dataset.*